# SOPQuery — RAG Pipeline
## Sprint 1 - Stage 1: PDF Ingestion
### Functions
- Successfully load PDFs without error.
- Verify content of extracted text that is readible, no error present and non-empty.


In [8]:
# import libraries
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

In [9]:
# load pdfs saved in data/FDA folder
# save all loaded documents into a variable: documents

pdf_path = Path("../data/FDA")

documents = []
for pdf in pdf_path.glob('*.pdf'):
    try:
        pdf_loader = PyPDFLoader(pdf)
        documents.extend(pdf_loader.load())
    
    # validate successful PDF ingestion
    except Exception as err:
        print(f'Failed to load {pdf.name}: {err}')

# check output to verify all 5 pdfs successfully loaded
expected = len(list(pdf_path.glob('*.pdf')))
actual = len(set(doc.metadata['source'] for doc in documents))
print(f'Expected PDFs: {expected}')
print(f'Succefully loaded PDFs: {actual}')
print(documents[0].metadata)

Expected PDFs: 5
Succefully loaded PDFs: 5
{'producer': 'iTextSharp 4.0.3 (based on iText 2.0.2)', 'creator': 'CommonLook Office-2.1.15.47', 'creationdate': '2026-02-03T08:15:53-05:00', 'author': 'CDRH FDA', 'keywords': '', 'moddate': '2026-02-03T09:10:12-05:00', 'nccl_app': 'Office', 'nccl_standard': 'Section 508; WCAG 2.0 AA; PDF/UA', 'nccl_status': 'Passed', 'subject': '', 'title': 'Computer Software Assurance for Production and Quality Management System Software', 'part': '1', 'source': '../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf', 'total_pages': 41, 'page': 0, 'page_label': '1'}


In [10]:
# print file name and number of pages of each files
file_info = set((doc.metadata['title'], doc.metadata['total_pages']) for doc in documents)

for title, pages in file_info:
    print(f'{title}: {pages} pages')

SOPP 8212: Breakthrough Therapy Products - Designation and Management: 25 pages
Computer Software Assurance for Production and Quality Management System Software: 41 pages
SOPP 8217: Administrative Process and Review Management of Investigational New Drug Applications: 31 pages
Investigating Out-of-Specification (OOS) Test Results for Pharmaceutical Production: 17 pages
SOPP 8201: Administrative Processing of Clinical Holds for Investigational New Drug Applications: 15 pages


In [11]:
# spot-check first 500 characters in first document
print(documents[0].page_content[:500])

Contains Nonbinding Recommendations
Computer Software Assurance for 
Production and Quality Management 
System Software
Guidance for Industry and 
Food and Drug Administration Staff 
Document issued on February 3, 2026.
This document supersedes “Computer Software Assurance for Production 
and Quality System Software,” issued September 24, 2025. 
For questions about this document regarding CDRH-regulated devices, contact the Compliance 
and Quality Staff at 301-796-5577 or by email at CaseforQual


## Sprint 1 - Stage 2: Chunking
### Functions
- Split documents into chunks that can be quickly retrieved and integrated into the model prompt.
- Configure chunk_size and overlap parameters
- Inspect chunk content that each chunk size is within the chunk_size character limit

In [12]:
# import library
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
# split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

# print the first 3 chunks
for i, chunk in enumerate(chunks[:3]):
    print(f'\n------ Chunk {i+1} ------')

    # Chunk size remains within 500 characters limit
    print(f'Characters: {len(chunk.page_content)}')
    print(f'Source: {chunk.metadata['source']}')
    print(f'Page: {chunk.metadata['page']}')
    
    # inspect chunk output
    print(chunk.page_content[:500])


------ Chunk 1 ------
Characters: 438
Source: ../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf
Page: 0
Contains Nonbinding Recommendations
Computer Software Assurance for 
Production and Quality Management 
System Software
Guidance for Industry and 
Food and Drug Administration Staff 
Document issued on February 3, 2026.
This document supersedes “Computer Software Assurance for Production 
and Quality System Software,” issued September 24, 2025. 
For questions about this document regarding CDRH-regulated devices, contact the Compliance

------ Chunk 2 ------
Characters: 459
Source: ../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf
Page: 0
and Quality Staff at 301-796-5577 or by email at CaseforQuality@fda.hhs.gov. For questions 
about this document regarding CBER-regulated devices, contact the Office of Communication, 
Outreach, and Development (OCOD) at 800-835-4709 or 240-402-8010, or by email at 
industry.biologics@fda.hhs.

## Sprint 1 - Stage 3: Embedding and Storing
### Functions
- Initialise embedding model and vector store
- Confirm that embeddings are stored in the vector store and document count matches

In [ ]:
# import libraries
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [15]:
# initialise embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name
)

# check if the vector store already exists on disk
vector_store= Chroma(
    persist_directory='../chroma_db',
    embedding_function=embedding_model
)

# create a vector store if not yet exist
if vector_store._collection.count() == 0:
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory='../chroma_db'
    )
    print('Created new vector store and persisted to disk.')
else:
    print('Loaded existing vector store from disk.')


# confirm chunk count matches to verify the database is persisted to disk for future use
print(f'Chunks written: {len(chunks)}')
print(f'Chunks in Chroma: {vector_store._collection.count()}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded existing vector store from disk.
Chunks written: 707
Chunks in Chroma: 707


## Sprint 1 - Stage 4: Similarity Search and Retrieval
### Functions
- Write query test set and run similarity search on the vector store
- Validate retrieval output that the most relevant chunks are returned

In [16]:
query = [
        'What should be included in a full-scale OOS investigation?',
         'During an OOS investigation, can the retest be done by the same analyst who performed the original test?',
         'What is the difference between process risk and medical device risk?',
         'Examples of unscripted testing.',
         'What is the timeframe for FDA to respond after receiving a complete response to a clinical hold?',
         'Who has the final authority to decide whether to issue a clinical hold on an IND?',
         'How long does CBER have to notify a sponsor whether their product has received breakthrough therapy designation?',
         'What happens if a sponsor does not respond to an Intent to Rescind Breakthrough Therapy Designation letter?',
         'When does an original IND go into effect if FDA does not notify the sponsor of a clinical hold?',
         'What is the review timeline for a breakthrough therapy designation request under this SOPP?'
         ]

for i, q in enumerate(query):
    # run similarity search and return the top 3 most relevant chunks
    result = vector_store.similarity_search(query=q,
                                        k=3)
    print(f'Query {i+1}: {q}\n')

    for doc in result:
        # inspect retrieval chunk putput
        print(f'Source: {doc.metadata["source"]}')
        print(f'\nPage number: {doc.metadata["page_label"]}\n')
        print(f'Chunk retrieved: \n{doc.page_content}')
        print('\n'+'-'*12)
    
    print('\n'+'='*30)

Query 1: What should be included in a full-scale OOS investigation?

Source: ../data/FDA/19287685_L2-OOS.pdf

Page number: 9

Chunk retrieved: 
testing results appear to be accurate, a full-scale OOS investigation using a predefined procedure 
should be conducted.  The objective of such an investigation should be to identify the root cause 
of the OOS result and take appropriate corrective action and preventive action.
7  A full-scale 
investigation should include a review of production and sampling procedures and will often 
include additional laboratory testing. Such investigations should be given the highest priority.

------------
Source: ../data/FDA/19287685_L2-OOS.pdf

Page number: 9

Chunk retrieved: 
manufacturer or at multiple manufacturing sites), all sites potentially involved should be 
included in the investigation.  Other potential problems should be identified and investigated. 
 
The records and documentation of the manufacturing process should be fully reviewed to 
det

### Test Log — Similarity Search Validation

| Query no. | Query | Source(s) | Top-3 Relevance | Notes |
|---|---|---|---|---|
| 1 | What should be included in a full-scale OOS investigation? | 19287685_L2-OOS.pdf | High | Top-2 chunks directly answer the query; top-3 is general overview, slightly less relevant. |
| 2 | Can the retest be done by the same analyst who performed the original test? | 19287685_L2-OOS.pdf | Medium | All three chunks relevant to retest procedure and analyst qualification, but the specific clause directly answering this question ("an analyst other than the one who performed the original test") falls outside the top-3 window.|
| 3 | What is the difference between process risk and medical device risk? | guidance-computer-software-assurance-production-quality-system.pdf | High | All three chunks directly define and contrast the two terms. |
| 4 | Examples of unscripted testing. | guidance-computer-software-assurance-production-quality-system.pdf | High | Top-1 and top-3 chunks overlap significantly in content, likely due to chunk size/overlap settings. |
| 5 | What is the timeframe for FDA to respond after receiving a complete response to a clinical hold? | SOPP-8201-Administrative-Processing-Clinical-Holds-INDs_V11.pdf | High | All three chunks relevant to clinical hold response timelines. |
| 6 | Who has the final authority to decide whether to issue a clinical hold on an IND? | SOPP-8201-Administrative-Processing-Clinical-Holds-INDs_V11.pdf | Medium | The source document does name the role (Clinical Division Director) in the sentence preceding the retrieved chunk, but the chunk boundary cuts it off, so the retrieved text only shows "serves as signatory authority" without the title. Likely a chunk-splitting artefact rather than a retrieval ranking issue; worth noting for chunk size/overlap tuning. |
| 7 | How long does CBER have to notify a sponsor whether their product has received breakthrough therapy designation? | SOPP-8212-Breakthrough-Therapy-Products-Designation-and-Management-V8-b.pdf | High | All three chunks relevant; top-1 gives the direct 60-day answer. |
| 8 | What happens if a sponsor does not respond to an Intent to Rescind Breakthrough Therapy Designation letter? | SOPP-8212-Breakthrough-Therapy-Products-Designation-and-Management-V8-b.pdf | High | All three chunks relevant to the rescind process and consequences. |
| 9 | When does an original IND go into effect if FDA does not notify the sponsor of a clinical hold? | SOPP-8217 and SOPP-8201 | High | Retrieved chunks from two different SOPPs both citing the same 30-day rule (21 CFR 312.40)|
| 10 | What is the review timeline for a breakthrough therapy designation request under this SOPP? | SOPP-8212-Breakthrough-Therapy-Products-Designation-and-Management-V8-b.pdf | High | All three chunks relevant to the request review process. |

**Summary:** 
- 9/10 queries returned highly relevant top-3 chunks.
- 1 query (Query 6) returned partially relevant results. 
- No retrieval failures observed. 
- Chunk overlap noted for Query 4 as a potential tuning consideration in later sprints.
- Cross-document references (SOPP-8217 and SOPP-8201) noted for Query 9, relevant for source citation design in sprint 2.

## Sprint 2 - Stage 1: Groq API set up
### Functions
- Set up GROQ API key
- Test API calling and confirm a response is returned

In [17]:
# import libraries
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

In [18]:
# load env file
load_dotenv()

api_key = os.getenv('GROQ_API_KEY')

# confirm API key is loaded successfully
print('API key loaded:', api_key is not None)

API key loaded: True


In [19]:
# initialise the llm
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=api_key
)

# test to invoke for a response
prompt = "What is GMP in pharmaceutical industry?"
response = llm.invoke(prompt)

print(response.content)

GMP stands for Good Manufacturing Practice in the pharmaceutical industry. It refers to a set of guidelines and regulations that ensure the quality, safety, and efficacy of pharmaceutical products, including medications, vaccines, and medical devices.

GMP guidelines cover various aspects of pharmaceutical manufacturing, including:

1. **Quality Control**: Ensuring that products meet the required standards of quality, purity, and potency.
2. **Quality Assurance**: Implementing a quality management system to ensure that products are consistently manufactured to meet the required standards.
3. **Facilities and Equipment**: Designing and maintaining facilities and equipment to prevent contamination and ensure proper sanitation.
4. **Personnel**: Training and qualifying personnel to perform their tasks correctly and safely.
5. **Processes**: Validating and controlling manufacturing processes to ensure consistency and reliability.
6. **Documentation**: Maintaining accurate and complete reco

## Sprint 2 - Stage 2: Prompt Construction
### Functions
- Design prompt template: system, human
- Format retrieved chunks into {context}
- Combine {context} and {query}
- Test designed prompt

> **Sprint 3 update:** Prompt template revised to add `chat_history` placeholder for FR-006 conversation history.

In [34]:
# import libraries
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Define system prompt for system instructions
system_prompt = (
    """
    You are a process expert in pharmaceutical industry, that answers questions from users based on the SOPs user uploaded.
    You always answer based on the contents of your assigned documents provided:

    Context:
    {context}

    If the user's question cannot be answered based on the documents you have access to, you must always address this clearly in your answer.
    """
)
# Create a chat prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{question}")
    ]
)

In [ ]:
# Test for user query and format retrieved chunks into context
user_query = "What should be included in a full-scale OOS investigation?"
result = vector_store.similarity_search(query=user_query, k=3)

# Create an empty list to store retrieved chunks
chunk_texts = []

for doc in result:
    chunk_texts.append(doc.page_content)

# join all chunks into one string
context = "\n\n".join(chunk_texts)

print(context)

# prompt structure verification with sample user query
prompt_test = prompt_template.invoke({"context": context, "question":user_query})
print(prompt_test.messages)

[SystemMessage(content="\n    You are a process expert in pharmaceutical industry, that answers questions from users based on the SOPs user uploaded.\n    You always answer based on the contents of your assigned documents provided:\n\n    Context:\n    testing results appear to be accurate, a full-scale OOS investigation using a predefined procedure \nshould be conducted.  The objective of such an investigation should be to identify the root cause \nof the OOS result and take appropriate corrective action and preventive action.\n7  A full-scale \ninvestigation should include a review of production and sampling procedures and will often \ninclude additional laboratory testing. Such investigations should be given the highest priority.\n\nmanufacturer or at multiple manufacturing sites), all sites potentially involved should be \nincluded in the investigation.  Other potential problems should be identified and investigated. \n \nThe records and documentation of the manufacturing process s

## Sprint 2 - Stage 3: Build RAG with LangChain
### Functions
- Create and connect retrieval prompt
- Build an LCEL retrieval chain (query → retrieve → prompt → generate)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Instantiate a retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

# define a function to format retreived chunks into a context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# create the LCEL retrieval chain
chain = (
    {"context":retriever|format_docs, "question":RunnablePassthrough()}
    |prompt_template
    |llm
    |StrOutputParser()
)

# MVP prototype
response = chain.invoke(user_query)
print(response)

According to the provided SOP, a full-scale OOS investigation should include:

1. A review of production and sampling procedures
2. Additional laboratory testing
3. A review of the records and documentation of the manufacturing process to determine the possible cause of the OOS result(s)
4. Identification of other potential problems and investigation of these issues
5. Involvement of all sites potentially affected (if the issue is related to multiple manufacturing sites)

The investigation should be timely, thorough, and well-documented, and should be given the highest priority. The objective of the investigation is to identify the root cause of the OOS result and take appropriate corrective action and preventive action.


In [ ]:
# run with all test queries
for i, q in enumerate(query):
    print(f'Q{i+1}: {q}')
    response = chain.invoke(q)
    print(f'LLM response: {response}\n')

Q1: What should be included in a full-scale OOS investigation?
LLM response: According to the provided SOP, a full-scale OOS investigation should include:

1. A review of production and sampling procedures
2. Additional laboratory testing
3. A review of the records and documentation of the manufacturing process to determine the possible cause of the OOS result(s)
4. Identification of other potential problems and investigation of these issues
5. Involvement of all sites potentially involved (if applicable)

The investigation should be timely, thorough, and well-documented, and should be given the highest priority. The objective of the investigation is to identify the root cause of the OOS result and take appropriate corrective action and preventive action.

Q2: During an OOS investigation, can the retest be done by the same analyst who performed the original test?
LLM response: According to the provided document, it is stated that "A second analyst performing a retest should be at least

### Test Log — RAG Pipeline end-to-end generation
8/10 queries returned accurate and relevant answers consistent with retrieved context.

Two queries (Q2, Q6) returned hedged or incomplete answers. On review, this was traced back to the same chunks already retrieved in Sprint 1 — the relevant content was incomplete.

Q10 also flagged a possible retrieval gap with table-formatted content (SOPP-8217 Appendix A) — to revisit during tuning.

No hallucinations were observed across all 10 queries.

## Sprint 2 - Stage 4: Source Citation
### Functions
- Append source citation to the end of the generated answer
- Test with sample queries to verify citation accuracy against retrieved chunks and are formatted correctly

In [25]:
# Extract source metadata
source_docs = retriever.invoke(user_query)

# Format the sources to get ready for appending to llm answer
sources = {}

for doc in source_docs:
    file_name = doc.metadata['source']
    page_number = doc.metadata['page_label']

    if file_name not in sources:
        sources[file_name] = set()

    sources[file_name].add(page_number)

llm_answer = chain.invoke(user_query)
print(llm_answer)

for path, pages in sources.items():
    file_name = os.path.basename(path)
    page_number = ', '.join(sorted(pages))
    print(f'(Source: {file_name}, pages: {page_number}')

A full-scale OOS (Out-of-Specification) investigation should include a review of production and sampling procedures, and will often include additional laboratory testing. The investigation should be given the highest priority and should be timely, thorough, and well-documented. 

The records and documentation of the manufacturing process should be fully reviewed to determine the possible cause of the OOS result(s). If the OOS result affects multiple sites (e.g., multiple manufacturing sites), all sites potentially involved should be included in the investigation. Other potential problems should be identified and investigated as well.

The objective of the investigation is to identify the root cause of the OOS result and take appropriate corrective action and preventive action.
(Source: 19287685_L2-OOS.pdf, pages: 4, 9


In [49]:
def source_citation(q):
    """Retrieve relevant chunks, generate an answer, and print with source citation."""
    
    # Extract source metadata
    # source_docs = retriever.invoke(q) *revisited and updated at stage 5 with similarity score threshold

    source_docs = vector_store.similarity_search_with_score(query=q, k=3)

    # check top-1 similarity score before processing
    if source_docs[0][1] >1:
        print("No relevant content found in the loaded SOP documents.")
        return

    # Format the sources to get ready for appending to llm answer
    sources = {}

    for doc, _ in source_docs:
        
        file_name = doc.metadata['source']
        page_number = doc.metadata['page_label']

        if file_name not in sources:
            sources[file_name] = set()

        sources[file_name].add(page_number)
    
    # Sprint 3 update: invoke chain_with_history to support conversation history (FR-006) and updated input key from "question" to "input" to align with chain_v2
    llm_answer = chain_with_history.invoke(
        {"input": q},
        config={"configurable": {"session_id": "session_1"}}
    )
    
    print(llm_answer)

    for path, pages in sources.items():
        file_name = os.path.basename(path)
        page_number = ', '.join(sorted(pages, key=lambda x:int(x)))
        print(f'(Source: {file_name}, pages: {page_number})')


### Test Log — Source Citation

- Source citation verified across all 10 test queries. All responses include correctly formatted citations (filename + deduplicated, numerically sorted page numbers). 
- Q9 confirmed multi-source citation working correctly (SOPP-8217 and SOPP-8201 listed separately).

## Sprint 2 - Stage 5: No-match handling
### Functions
- Define no-match criteria and design fallback response
- Test with off-topic queries to verify no-match handling and fallback response

In [28]:
# check similarity scores for an on-topic query
scores_on_topic = vector_store.similarity_search_with_score(
    query=query[0],
    k=3
)

for doc, score in scores_on_topic:
    print(f"Score: {score:.4f} | Page: {doc.metadata['page_label']} | {doc.page_content[:100]}")

Score: 0.5098 | Page: 9 | testing results appear to be accurate, a full-scale OOS investigation using a predefined procedure 

Score: 0.5267 | Page: 9 | manufacturer or at multiple manufacturing sites), all sites potentially involved should be 
included
Score: 0.7296 | Page: 4 | the guidance discusses how to investigate OOS test results, including the responsibilities of  
labo


In [29]:
# check similarity scores for an off-topic query
off_topic = 'What are the side effects of Ozempic?'

scores_off_topic = vector_store.similarity_search_with_score(
    query=off_topic,
    k=3
)

for doc, score in scores_off_topic:
    print(f"Score: {score:.4f} | Page: {doc.metadata['page_label']} | {doc.page_content[:100]}")

Score: 1.2839 | Page: 2 | with 1 or more other drugs, to treat a serious or life threatening disease or 
condition and prelimi
Score: 1.3939 | Page: 24 | Center for Biologics Evaluation and Research  SOPP 8212 Appendix A 
Page 4 of 5 
Nonclinical Pharmac
Score: 1.4611 | Page: 24 | Examples of such studies include: 
a. Subacute and chronic toxicology and associated toxicokinetics 


In [30]:
# test with off-topic query to verify implementation of fallback response
print(f'Off-topic query: {off_topic}\n')
source_citation(off_topic)
print('-'*50)

# test with on-topic query ensure no negative impact following the implementationß
print(f'\nOn-topic query: {query[0]}\n')
source_citation(query[0])

Off-topic query: What are the side effects of Ozempic?

No relevant content found in the loaded SOP documents.
--------------------------------------------------

On-topic query: What should be included in a full-scale OOS investigation?

A full-scale OOS (Out-of-Specification) investigation should include the following:

1. A review of production and sampling procedures.
2. Additional laboratory testing.
3. A review of the records and documentation of the manufacturing process to determine the possible cause of the OOS result(s).
4. Identification and investigation of other potential problems.
5. Inclusion of all sites potentially involved in the investigation, if applicable (e.g., multiple manufacturing sites).

The investigation should be timely, thorough, and well-documented, and should be given the highest priority. The objective of the investigation is to identify the root cause of the OOS result and take appropriate corrective action and preventive action.
(Source: 19287685_L2-O

### Test Log — No-match handling

Implemented similarity score threshold of 1.0 on top-1 retrieved chunk to detect off-topic queries before invoking the LLM. Threshold was determined by comparing on-topic scores (0.51–0.73) against off-topic scores (1.28–1.46).

- Off-topic: "What are the side effects of Ozempic?" was used to validate the implementation. Fallback message triggered, LLM not invoked, no source citation displayed.

In [32]:
# check response time (target <= 20s)
import time

start = time.time()
source_citation(query[0])
end = time.time()

print(f'\nResponse time:{end - start:.2f} seconds')

A full-scale OOS (Out-of-Specification) investigation should include several key components. According to the provided documents, a full-scale investigation should consist of:

1. A review of production and sampling procedures.
2. Additional laboratory testing.
3. A review of the records and documentation of the manufacturing process to determine the possible cause of the OOS result(s).
4. A timely, thorough, and well-documented investigation.

It is also recommended that if the issue is related to a specific product manufactured at multiple sites, all sites potentially involved should be included in the investigation. Furthermore, other potential problems should be identified and investigated.

The investigation should be given the highest priority, and its objective should be to identify the root cause of the OOS result and take appropriate corrective action and preventive action.
(Source: 19287685_L2-OOS.pdf, pages: 4, 9)

Response time:1.03 seconds


## Sprint 3 - Stage 1: Conversation History
### Function
- Implement conversation history that allows user to ask follow-up queries within the same chat window

In [ ]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory


In [50]:
# design new prompt in terms of query contextualisation
rephrase_sys_prompt = "Given the chat history and the latest user question, reformulate the question to be standalone and self-contained. Do not answer the question."

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", rephrase_sys_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)

prompt_template_v2 = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}")
    ]
)

# create new chain
chain_v2 = (
    {"context": history_aware_retriever | format_docs, "input": RunnablePassthrough()}
    | prompt_template_v2
    | llm
    | StrOutputParser()
)

# create an empty dict to store chat message history
store={}

def get_by_session_id(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    return store[session_id]

# build chain with message history
chain_with_history = RunnableWithMessageHistory(
    chain_v2,
    get_by_session_id,
    input_messages_key="input",
    history_messages_key="chat_history"
)

/Users/janelam/sop-qa-agent/venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [53]:
# Sprint 3 regression test: verify source_citation() still works after FR-006 update
# Verify multi-source citation is being handled as expected
source_citation(query[8])

# verify source citation against all test queries
for i, q in enumerate(query):
    print(f'Q{i+1}) {q}')
    source_citation(q)
    print('-'*40)

An original IND goes into effect 30 calendar days after FDA receives the IND, unless FDA notifies the sponsor that the clinical investigations described in the IND are subject to a clinical hold. This is stated in 21 CFR 312.40(b) and also mentioned in the provided SOPP 8201. According to the SOP, if FDA does not notify the sponsor of a clinical hold, the IND will go into effect after the 30-day period.
(Source: SOPP-8217-Administrative-Processing-Review-Management-Procedures-IND-Applications_V6.pdf, pages: 14, 16)
(Source: SOPP-8201-Administrative-Processing-Clinical-Holds-INDs_V11.pdf, pages: 11)
Q1) What should be included in a full-scale OOS investigation?
A full-scale OOS investigation should include the following key points:
* A review of production and sampling procedures and may include additional laboratory testing.
* The investigation should be timely, thorough, and well-documented, with a full review of the manufacturing process records and documentation to determine the pos

In [51]:
# initial query
print(chain_with_history.invoke(
    {"input":user_query},
    config={"configurable":{"session_id":"session_1"}}
    )
)

A full-scale OOS investigation should include a review of production and sampling procedures and will often include additional laboratory testing. It should be a timely, thorough, and well-documented investigation. The records and documentation of the manufacturing process should be fully reviewed to determine the possible cause of the OOS result(s). If the OOS result affects multiple sites (e.g., a manufacturer or at multiple manufacturing sites), all sites potentially involved should be included in the investigation. Other potential problems should be identified and investigated.


In [52]:
#Test with follow-up query
follow_up_query = "Can you summarise that in 3 bullet points?"
print(f'Follow-up query: {follow_up_query}\n\
      {chain_with_history.invoke(
        {"input":follow_up_query},
        config={"configurable":{"session_id":"session_1"}}
      )}'
)

Follow-up query: Can you summarise that in 3 bullet points?
      Here are 3 bullet points summarizing the key points of a full-scale OOS investigation:
* A full-scale OOS investigation should include a review of production and sampling procedures and may include additional laboratory testing.
* The investigation should be timely, thorough, and well-documented, with a full review of the manufacturing process records and documentation to determine the possible cause of the OOS result(s).
* All sites potentially involved should be included in the investigation if the OOS result affects multiple sites, and other potential problems should also be identified and investigated.


### Test log - conversation history
- Initial query: OOS investigation
- Follow-up query: Summarise that in 3 bullet points
- Result: Pass. LLM correctly referenced prior context in the follow-up response
### Test log - regression testing (from sprint 2)
- Q2: improved retrieval captures answer-bearing clause
- Q6: no change, chunk boundary issue persists
- Q10: slight improvement, able to retrieve relevant text (role title), however the table content gap remains.
- no regression issues observed with other queries

## Sprint 3 - Stage 2: RAGAS evaluation
### Function
- Evaluate RAG piepline with RAGAS framework
- Assess retrieval and generation quality with faithfulness and context relevance metrics

In [ ]:

# create an empty list to store evalutaion dataset
eval_data = []

for q in query:
    # retrieve context
    source_docs = vector_store.similarity_search_with_score(query=q, k=3)
    contexts = [doc.page_content for doc, _ in source_docs]

    # generate answer
    answer = chain_with_history.invoke(
        {"input":q},
        config={"configurable":{"session_id":"ragas_eval"}}
    )

    # add each question, answer，retrieved context and ground truth list into eval_data
    eval_data.append({
        "question":q,
        "answer":answer,
        "contexts":contexts,
    })

In [68]:
# import libraries
import instructor
# from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness, ContextRelevance
from groq import AsyncGroq
from ragas.llms.base import InstructorLLM

# setu RAGAS llm evaluator
ragas_client = instructor.from_groq(AsyncGroq(api_key=api_key))
ragas_llm = InstructorLLM(
    model="llama-3.3-70b-versatile",
    provider="groq",
    client=ragas_client
)

# create metrics
faithfulness_scorer = Faithfulness(llm=ragas_llm)
context_relevance_scorer = ContextRelevance(llm=ragas_llm)

# run evaluation
# create an empty list to store evaluation results
eval_results = []

for item in eval_data:
    f_score = await faithfulness_scorer.ascore(
        user_input=item["question"],
        response=item["answer"],
        retrieved_contexts=item["contexts"]
    )

    cr_score = await context_relevance_scorer.ascore(
        user_input=item["question"],
        retrieved_contexts=item["contexts"]        
    )

    eval_results.append({
        "question":item["question"],
        "faithfulness":f_score.value,
        "context_relevance":cr_score.value,
    })

In [ ]:
print(eval_results)

[{'question': 'What should be included in a full-scale OOS investigation?', 'faithfulness': 1.0, 'context_relevance': 1.0}, {'question': 'During an OOS investigation, can the retest be done by the same analyst who performed the original test?', 'faithfulness': 0.6, 'context_relevance': 1.0}, {'question': 'What is the difference between process risk and medical device risk?', 'faithfulness': 0.7142857142857143, 'context_relevance': 1.0}, {'question': 'Examples of unscripted testing.', 'faithfulness': 0.7142857142857143, 'context_relevance': 1.0}, {'question': 'What is the timeframe for FDA to respond after receiving a complete response to a clinical hold?', 'faithfulness': 1.0, 'context_relevance': 1.0}, {'question': 'Who has the final authority to decide whether to issue a clinical hold on an IND?', 'faithfulness': 0.0, 'context_relevance': 0.5}, {'question': 'How long does CBER have to notify a sponsor whether their product has received breakthrough therapy designation?', 'faithfulnes

In [ ]:
for i in eval_results:
    print(i["question"])
    print(f"Faithfulness: {i["faithfulness"]}")
    print(f"Context relevance: {i["context_relevance"]}")

    if i["faithfulness"] >= 0.7 and i["context_relevance"] >= 0.7:
        print("Pass")
    else:
        print("Fail")

# avergae faithfulness and context relevance scores
avg_f_score = sum([i["faithfulness"] for i in eval_results]) / len(eval_results)
avg_cr_score = sum([i["context_relevance"] for i in eval_results]) / len(eval_results)

print(f"\nAverage faithfulness score: {avg_f_score}")
print(f"\nAverage context relevance score: {avg_cr_score}")

### Test log - RAGAS evaluation scores

| Query no. | Faithfulness | Context Relevance | Result (Pass/Fail) |
|---|---|---|---|
|1|1.0|1.0|Pass|
|2|0.6|1.0|Fail|
|3|1.0|0.71|Pass|
|4|1.0|0.71|Pass|
|5|1.0|1.0|Pass|
|6|0.0|0.5|Fail|
|7|1.0|1.0|Pass|
|8|0.6|1.0|Fail|
|9|0.67|1.0|Fail|
|10|0.0|0.5|Fail|

- Average faithfulness and context relevance scores are x and y, respectively.
- The passing criteria for each query is based on a context relevance score of ≥ 0.7 and answer faithfulness score of ≥ 0.7. 
- Low faithfulness and context relevance scores for Q6 and Q10 were expected, reflecting to the known retrieval gap issue.
- 50% passing rate calls for parameter tuning in stage 3.